<a href="https://colab.research.google.com/github/joaocanaslopes/Assignments_ML/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ML Exercise: Fine-tuning a CNN on MNIST with PyTorch

This notebook addresses the challenge of fine-tuning a Convolutional Neural Network (CNN) for the MNIST dataset using PyTorch. The goal is to train a model, save its weights, and prepare it for deployment on Hugging Face Spaces with a Gradio interface.

### 1. Setup and Imports

First, we'll install PyTorch and torchvision, then import the necessary libraries.

In [1]:
# Install PyTorch and torchvision if not already installed (usually pre-installed in Colab)
# !pip install torch torchvision torchaudio

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

### 2. Data Loading and Preprocessing

We'll load the MNIST dataset and apply transformations suitable for a CNN, such as converting images to tensors and normalizing them.

In [2]:
# Define transformations for the MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(), # Convert images to PyTorch tensors
    transforms.Normalize((0.1307,), (0.3081,)) # Normalize pixel values (mean and std for MNIST)
])

# Download and load the MNIST training and test datasets
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Define data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training data size: {len(train_dataset)}")
print(f"Test data size: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 44.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.5MB/s]


Training data size: 60000
Test data size: 10000


### 3. Define the Convolutional Neural Network (CNN) Model

Here, we define a simple CNN architecture suitable for the MNIST dataset. For this task, we will create a custom CNN. If a specific pre-trained model from `torchvision` (like ResNet or VGG) is desired, additional steps would be needed to adapt it for single-channel (grayscale) 28x28 images and 10 output classes.

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1) # 1 input channel (grayscale), 32 output channels
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Output size: 14x14

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 32 input channels, 64 output channels
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Output size: 7x7

        # Fully connected layer
        # Input features: 64 channels * 7 * 7 (from previous pooling layer)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10) # 10 output classes for digits 0-9

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7) # Flatten the tensor for the fully connected layer
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and move it to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

print(f"Model will be trained on: {device}")

Model will be trained on: cpu


### 4. Training the Model

We will define the loss function, optimizer, and implement the training loop for a few epochs.

In [4]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Number of epochs to train
num_epochs = 5 # As requested, 'a couple of epochs'

# Training loop
print("Starting training...")
for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(data)
        loss = criterion(outputs, target)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete!")

Starting training...
Epoch [1/5], Loss: 0.1326
Epoch [2/5], Loss: 0.0432
Epoch [3/5], Loss: 0.0293
Epoch [4/5], Loss: 0.0209
Epoch [5/5], Loss: 0.0167
Training complete!


### 5. Evaluate the Model

Let's evaluate the model's performance on the test set.

In [5]:
model.eval() # Set the model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations during evaluation
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy of the model on the {total} test images: {accuracy:.2f}%")

Accuracy of the model on the 10000 test images: 98.48%


### 6. Save Model Weights

Finally, we'll save the trained model's state dictionary to a `.pth` file, as requested. This file can then be used to load the model for inference or deployment.

In [6]:
model_save_path = 'mnist_cnn_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

Model weights saved to mnist_cnn_model.pth


### 7. Preparando para Implantação no Hugging Face Spaces com Gradio

Para implantar o modelo treinado no Hugging Face Spaces usando Gradio, precisaremos de dois arquivos principais: `app.py` (que definirá a interface Gradio e carregará o modelo) e `requirements.txt` (que listará as dependências Python).

#### `app.py` (código para a aplicação Gradio)

Este script carregará o modelo salvo e definirá uma interface Gradio para classificar dígitos manuscritos. Incluirá a funcionalidade de desenhar dígitos. Você precisará copiar este conteúdo para um arquivo chamado `app.py` no seu Hugging Face Space.

In [ ]:
%%writefile app.py

import torch
import torch.nn as nn
from torchvision import transforms
import gradio as gr
from PIL import Image
import numpy as np

# 1. Definir a mesma arquitetura do modelo (COMO USADO NO TREINAMENTO)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. Carregar o modelo treinado
# Garanta que o modelo seja carregado para a CPU, pois nem todos os Spaces terão GPU
device = torch.device('cpu')
model = SimpleCNN().to(device)
model.load_state_dict(torch.load('mnist_cnn_model.pth', map_location=device))
model.eval() # Definir para modo de avaliação

# 3. Definir transformações para a imagem de entrada (deve ser a mesma do treinamento)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), # Garantir que a imagem seja em tons de cinza
    transforms.Resize((28, 28)), # Redimensionar para 28x28 pixels
    transforms.ToTensor(), # Converter para tensor PyTorch
    transforms.Normalize((0.1307,), (0.3081,)) # Normalizar com os mesmos valores
])

# 4. Função de previsão
def predict_digit(image):
    if image is None:
        return {str(i): 0.0 for i in range(10)}

    # Converter a imagem PIL para um array numpy antes de aplicar as transformações
    # Gradio retorna a imagem como numpy array ou PIL Image dependendo do componente
    # Se for numpy, converte para PIL para o transform.Compose
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image.astype('uint8'), 'L') # 'L' para grayscale

    image_tensor = transform(image).unsqueeze(0).to(device) # Adicionar dimensão de batch

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]

    # Retornar um dicionário de probabilidades
    return {str(i): prob.item() for i, prob in enumerate(probabilities)}

# 5. Configurar a interface Gradio
# Criar um componente de desenho para a entrada
drawing_canvas = gr.Sketchpad()

# Adicionar exemplos
example_images = [
    'example_0.png', # Você precisaria criar essas imagens e fazer upload para o HF Space
    'example_1.png',
    'example_2.png',
    # ... adicione mais exemplos se desejar
]

# Se você não tiver imagens de exemplo, pode remover a linha 'examples' do gr.Interface
# ou criar algumas manualmente com um editor de imagem e fazer upload para o Hugging Face Space

if __name__ == '__main__':
    # Interface para desenhar
    drawing_interface = gr.Interface(
        fn=predict_digit,
        inputs=drawing_canvas,
        outputs=gr.Label(num_top_classes=10),
        title="Classificador de Dígitos MNIST",
        description="Desenhe um dígito (0-9) no quadro e o modelo predizerá qual é.",
        live=True
    )

    # Interface para upload de imagem (opcional, pode ser combinada com o desenho ou separada)
    upload_interface = gr.Interface(
        fn=predict_digit,
        inputs=gr.Image(type="pil", label="Carregar Imagem de Dígito"),
        outputs=gr.Label(num_top_classes=10),
        title="Classificador de Dígitos MNIST (Upload)",
        description="Carregue uma imagem de um dígito manuscrito para classificação.",
        # examples=[example_images] # Descomente se tiver exemplos
    )

    # Combinar as interfaces em um TabbedInterface
    gr.TabbedInterface([
        drawing_interface,
        upload_interface
    ], ["Desenhar", "Upload"]).launch()

#### `requirements.txt` (dependências Python)

Este arquivo lista todas as bibliotecas Python que precisam ser instaladas no ambiente do Hugging Face Space para que sua aplicação Gradio funcione. Você precisará copiar este conteúdo para um arquivo chamado `requirements.txt` no seu Hugging Face Space.

In [ ]:
%%writefile requirements.txt
torch
torchvision
Pillow # Necessário para processamento de imagens PIL

### 8. Hugging Face Space Deploy

Meu modelo foi implantado com sucesso no Hugging Face Spaces e pode ser acessado através do seguinte link:
[Meu MNIST CNN Gradio App](https://huggingface.co/spaces/Rainozita/ML_classes)